# ETF Factor Momentum: Alpha Validation Under Implementation Frictions

This notebook implements an ETF-based cross-sectional factor-momentum research pipeline. It is designed as an **alpha-validation** project rather than as a claim that the retained strategy is a production-ready alpha.

The workflow tests whether a simple Arnott-style factor-momentum signal survives translation into a tradable ETF implementation with next-day execution, transaction-cost proxies, passive benchmarking, signal-permutation nulls, dependence-aware bootstrap inference, spanning regressions and data-snooping controls.

**Headline finding.** Top-factor momentum improves realized drawdown-adjusted performance and beats structurally identical random-selection nulls, but the effect is largely defensive, substantially replicated by static factor/market exposures, and does not survive the full paired-bootstrap / SPA evidence as a robust standalone alpha.


## Setup

Install dependencies from `requirements.txt` before running the notebook.

```bash
pip install -r requirements.txt
```

The notebook imports all reusable research functions from `factor_momentum_core.py`.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from factor_momentum_core import *
from factor_momentum_core import _metric_sharpe, _metric_cagr

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


## Configuration

- `DATA_SOURCE`: `"hdf"` reproduces the original master-project panel; `"yahoo"` is the public-data workflow.
- `IMPLEMENTATION_LAG`: `1` = trade at the close of the day after the signal (next-day); `0` = trade at the signal-date close (legacy).
- Transaction costs: smoothed one-way half-spread proxies (AR/CS) or a flat `TC_FIXED_BPS`, applied at monthly rebalances.
- `RUN_BENCHMARK` adds `SPY` as a passive market benchmark.

In [ ]:

# ============================================================
# USER CONFIGURATION
# ============================================================

DATA_SOURCE = "yahoo"           # "yahoo" or "hdf"
HDF_PATH = None                 # e.g. "data/factor_etfs.h5" if DATA_SOURCE == "hdf"
HDF_KEY = "/FactorETFs"
START_DATE = "2000-01-01"
VARIANT = "6-1"                 # "6-1" or "12-1"

# --- Implementation timing ---
# 1 = genuine next-day implementation: trade at the close of the trading day AFTER
#     the signal date; new weights first earn on the following day. This removes the
#     ~1-day look-ahead that exists when new weights earn the trade-day return.
# 0 = legacy "execute at the signal-date close" convention.
IMPLEMENTATION_LAG = 1

# --- Historical risk-free rate ---
USE_HISTORICAL_RF = True
RF_SOURCE = "DGS3MO"            # "DGS3MO", "DTB3", or "SOFR" (SOFR only from 2018-04-03)
RF_DAY_COUNT = 360
RF_LAG_DAYS = 1

# --- Market benchmark ---
RUN_BENCHMARK = True
BENCH_TICKER = "SPY"

# --- Transaction costs ---
RUN_TRANSACTION_COSTS = True
TC_ESTIMATOR = "AR"             # primary estimator for plots/tables: "AR", "CS" or "FIXED"
RUN_TC_ESTIMATOR_COMPARISON = True
TC_ESTIMATORS_TO_COMPARE = ("AR", "CS", "FIXED")   # FIXED = flat bps sanity check
TC_FIXED_BPS = 2.0              # one-way cost in bps for the FIXED estimator
TC_LOOKBACK_DAYS = 21
TC_AGG = "median"               # "median" or "mean"
TC_SPREAD_CLIP_UPPER = 0.10
TC_CHARGE_INITIAL_TRADE = True

# --- Bootstrap ---
RUN_ACF_BLOCK_SELECTION = True
ACF_CANDIDATES = (5, 10, 20, 30)
ACF_NLAGS = 20
ACF_N_BOOT = 300

RUN_BOOTSTRAP = True
N_BOOT = 1000
AVG_BLOCK_LEN = 10              # used when RUN_ACF_BLOCK_SELECTION = False
BOOTSTRAP_SEED = 7

# --- Random-selection (signal-permutation) null ---
RUN_RANDOM_NULL = True
N_RANDOM = 1000
RANDOM_NULL_SEED = 12345

# metrics used for the paired-difference / engine tables
PAIRED_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

# --- Alternative weighting (separate selection from concentration) ---
RUN_ALT_WEIGHTING = True
ALT_WEIGHT_LOOKBACK = 63

# --- Serial-dependence diagnostics (motivate the bootstrap engine) ---
RUN_DEPENDENCE_DIAGNOSTICS = True
DIAG_LAGS = 10

# --- Spanning regression (Newey-West alpha vs the factor ETFs + SPY) ---
RUN_SPANNING = True

# --- Sharpe-ratio inference (Ledoit-Wolf delta-Sharpe; PSR) ---
RUN_SHARPE_INFERENCE = True
LW_N_BOOT = 4999

# --- Break-even one-way transaction cost ---
RUN_BREAKEVEN = True
BREAKEVEN_C_MAX_BPS = 200.0

# --- BCa intervals on the realized P&L ---
RUN_BCA = True
BCA_N_BOOT = 2000

# --- Regime / subsample stability ---
RUN_REGIME = True
REGIME_VOL_LOOKBACK = 21
REGIME_N = 2

# --- Dependence-aware bootstrap engines (FHS AR-GJR-GARCH; VAR-sieve) ---
RUN_ENGINE_BOOTSTRAP = True
ENGINES = ("fhs", "var_sieve")     # subset of these
ENGINE_N_SIMS = 500
ENGINE_AR_LAGS = 1
ENGINE_VAR_LAGS = 1
ENGINE_FLAT_COST_BPS = 2.0         # simulated panels carry no OHLC -> flat-bps costs
ENGINE_SEED = 21

# --- Multiple testing across the specification grid + Deflated Sharpe ---
RUN_MULTIPLE_TESTING = True
MT_BLOCK_SIZE = 10
MT_N_BOOT = 1000
MT_SEED = 11

# --- Long-short academic factor comparison (Fama-French via pandas_datareader) ---
RUN_LONGSHORT = True
LONGSHORT_CSV_PATH = None         # optional fallback: CSV of daily factor returns (decimal)
LONGSHORT_COST_BPS = 0.0          # factor returns are notional; set > 0 to stress turnover

# --- True out-of-sample walk-forward over the specification grid ---
RUN_WALK_FORWARD = True
WF_MIN_TRAIN_DAYS = 756           # ~3y initial in-sample window
WF_RESELECT_EVERY_DAYS = 63       # re-pick the best spec ~quarterly

# --- Rank / quantile weighting (+ extended-universe hook) ---
RUN_QUANTILE_WEIGHTING = True
QUANTILE_TOP_FRAC = 0.4
EXTENDED_TICKERS = None           # set TICKERS = EXTENDED_TICKERS (>5 distinct single-factor ETFs) and re-run

# --- Conditional (regime) spanning, rolling alpha, static replication, selection mechanism ---
RUN_CONDITIONAL_SPANNING = True
COND_VOL_LOOKBACK = 21
RUN_SELECTION_MECHANISM = True

## Load data, build risk-free + benchmark, estimate costs, run the backtest

1. load the raw factor-ETF panel and restrict to the **common-data sample**;
2. build the **historical daily risk-free** series (FRED `DGS3MO`, falling back to Yahoo `^IRX` then to zero if both are unreachable);
3. pull the **SPY** benchmark;
4. estimate daily cost panels under **AR**, **CS** and a **flat-bps** alternative;
5. run the main backtest under the primary estimator and `IMPLEMENTATION_LAG`;
6. report **gross**/**net** metrics (incl. benchmark), turnover (two-way and one-way), and weights.

In [ ]:

# ============================================================
# LOAD DATA
# ============================================================

bundle = None
if DATA_SOURCE == "hdf":
    if HDF_PATH is None:
        raise ValueError("Set HDF_PATH before using DATA_SOURCE='hdf'.")
    r_raw = load_factor_etf_returns_hdf(HDF_PATH, key=HDF_KEY)
    if RUN_TRANSACTION_COSTS:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))
elif DATA_SOURCE == "yahoo":
    bundle = download_factor_etf_bundle(start=START_DATE)
    r_raw = bundle["rets"]
else:
    raise ValueError("DATA_SOURCE must be either 'hdf' or 'yahoo'.")

r_raw = r_raw.copy()
r_common = r_raw.dropna(how="any").copy()

print(f"Loaded raw returns panel {r_raw.shape} from {r_raw.index.min().date()} to {r_raw.index.max().date()}.")
display(r_raw.head())
print(f"Common-data backtest sample: {r_common.shape} from {r_common.index.min().date()} to {r_common.index.max().date()}.")

# ============================================================
# MARKET BENCHMARK (passive)
# ============================================================
# EW of the five factor ETFs is itself an active factor bet, not a neutral
# "do nothing". SPY is added as a passive market benchmark for context.
bench_ret = None
if RUN_BENCHMARK:
    bench_ret = download_benchmark_returns(BENCH_TICKER, start=str(r_common.index.min().date()))
    bench_ret = bench_ret.reindex(r_common.index).dropna()
    print(f"Benchmark: {BENCH_TICKER} (buy-and-hold), {len(bench_ret)} aligned observations.")

# ============================================================
# HISTORICAL RISK-FREE SERIES
# ============================================================
if USE_HISTORICAL_RF:
    rf_daily, rf_source_used = build_rf_with_fallback(
        r_common.index, rf_source=RF_SOURCE, day_count=RF_DAY_COUNT, lag_days=RF_LAG_DAYS)
    print(f"Risk-free source in use: {rf_source_used}")
    display(rf_daily.to_frame("rf_daily").head())
else:
    rf_daily = pd.Series(0.0, index=r_common.index, name="rf_daily")
    rf_source_used = "ZERO"
    print("Historical risk-free disabled: using rf_daily = 0.0.")

# ============================================================
# ESTIMATE DAILY TRANSACTION-COST PANELS (AR / CS / FIXED)
# ============================================================
cost_oneway_daily_main = None
spread_full_panels = {}
cost_oneway_panels = {}
tc_mean_table = None
tc_corr_overall = np.nan
tc_corr_by_etf = None

estimators_requested = [TC_ESTIMATOR]
if RUN_TC_ESTIMATOR_COMPARISON:
    estimators_requested = list(dict.fromkeys([TC_ESTIMATOR] + list(TC_ESTIMATORS_TO_COMPARE)))

if RUN_TRANSACTION_COSTS:
    if bundle is None:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))
    px_close = bundle["close"].reindex(r_common.index)
    px_high = bundle["high"].reindex(r_common.index)
    px_low = bundle["low"].reindex(r_common.index)

    for est in estimators_requested:
        est = est.upper()
        if est == "AR":
            spread_full = abdi_ranaldo_panel(px_high, px_low, px_close,
                                              tickers=FACTOR_NAMES, clip_upper=TC_SPREAD_CLIP_UPPER)
            spread_full_panels[est] = spread_full.reindex(r_common.index)
            cost_oneway_panels[est] = spread_panel_to_one_way_cost(spread_full).reindex(r_common.index)
        elif est == "CS":
            spread_full = corwin_schultz_panel(px_high, px_low,
                                               tickers=FACTOR_NAMES, clip_upper=TC_SPREAD_CLIP_UPPER)
            spread_full_panels[est] = spread_full.reindex(r_common.index)
            cost_oneway_panels[est] = spread_panel_to_one_way_cost(spread_full).reindex(r_common.index)
        elif est == "FIXED":
            cost_oneway_panels[est] = constant_one_way_cost_panel(r_common.index, FACTOR_NAMES, TC_FIXED_BPS)
        else:
            raise ValueError("Each transaction-cost estimator must be 'AR', 'CS' or 'FIXED'.")

    cost_oneway_daily_main = cost_oneway_panels[TC_ESTIMATOR.upper()]
    print(f"Primary transaction-cost estimator: {TC_ESTIMATOR.upper()}")

    tc_mean_table = pd.concat(
        {est: cost_oneway_panels[est].mean().rename("one_way_cost") for est in cost_oneway_panels}, axis=1)
    print("Mean estimated one-way cost by ETF (bps):")
    display((tc_mean_table * 1e4).round(2))

    if {"AR", "CS"}.issubset(set(spread_full_panels.keys())):
        tc_corr_overall = float(spread_full_panels["AR"].stack().corr(spread_full_panels["CS"].stack()))
        tc_corr_by_etf = pd.Series(
            {fac: float(spread_full_panels["AR"][fac].corr(spread_full_panels["CS"][fac]))
             for fac in FACTOR_NAMES}, name="AR_vs_CS_corr").to_frame()
        print("AR vs CS full-spread correlation (overall):", round(tc_corr_overall, 4))
        display(tc_corr_by_etf.round(4))

# ============================================================
# RUN PRIMARY STRATEGY
# ============================================================
out = run_factor_mom_from_returns(
    r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_daily_main,
    cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
    charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
)
rets_gross = out["rets_gross"]
rets_net = out["rets_net"]
signal = out["signal"]
avg_daily_w = out["avg_daily_weights"]
avg_entry_w = out["avg_entry_weights"]
turnover_summary = out["turnover_summary"]
sample_info = out["sample_info"]

# attach the passive benchmark as an extra column for side-by-side reporting
rets_gross_bench = rets_gross.copy()
rets_net_bench = rets_net.copy()
if bench_ret is not None:
    b = bench_ret.reindex(rets_gross.index).rename(BENCH_TICKER)
    rets_gross_bench = pd.concat([rets_gross, b], axis=1)
    rets_net_bench = pd.concat([rets_net, b], axis=1)   # buy-and-hold: gross == net

actual_metrics_gross = perf_table(rets_gross_bench, rf_daily=rf_daily)
actual_metrics_net = perf_table(rets_net_bench, rf_daily=rf_daily)

print("Backtest sample information:")
display(pd.Series(sample_info, name="value").to_frame())
print(f"Implementation lag: {IMPLEMENTATION_LAG} trading day(s).")

print("Gross performance metrics (incl. benchmark):")
display(actual_metrics_gross.round(4))
print(f"Net performance metrics — primary estimator ({TC_ESTIMATOR.upper()}), incl. benchmark:")
display(actual_metrics_net.round(4))
print("Rebalancing / turnover summary (two-way and one-way):")
display(turnover_summary.round(4))
print("Average daily weights:")
display(avg_daily_w.round(4))

# ============================================================
# IN-SAMPLE NET COMPARISON: AR vs CS vs FIXED
# ============================================================
comparison_metrics_net = {}
comparison_turnover = {}
if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
    for est in estimators_requested:
        est_out = run_factor_mom_from_returns(
            r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_panels[est],
            cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
            charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        comparison_metrics_net[est] = perf_table(est_out["rets_net"], rf_daily=rf_daily)
        comparison_turnover[est] = est_out["turnover_summary"]
    print("Net in-sample metrics — estimator comparison (AR vs CS vs FIXED):")
    display(pd.concat(comparison_metrics_net, names=["Estimator", "Strategy"]).round(4))

## Cumulative wealth (gross vs net, with benchmark)

In [ ]:
wealth_gross = wealth_index(rets_gross_bench)
wealth_net = wealth_index(rets_net_bench)

ax = wealth_gross.plot(figsize=(11, 5), title=f"Factor Momentum ({VARIANT}) vs benchmark — gross")
ax.set_ylabel("Wealth index"); plt.show()

ax = wealth_net.plot(figsize=(11, 5),
                     title=f"Factor Momentum ({VARIANT}) vs benchmark — net ({TC_ESTIMATOR.upper()})")
ax.set_ylabel("Wealth index"); plt.show()

## Optional data-driven block-length selection

Average block length chosen by matching the autocorrelation of **absolute returns** of `MomTop1_80_20`. When costs are on, selection uses the **gross** series while the bootstrap resamples the aligned cost panel jointly with returns.

In [ ]:
if RUN_ACF_BLOCK_SELECTION:
    bl = choose_block_length_by_acf_matching(
        rets_gross["MomTop1_80_20"], candidates=ACF_CANDIDATES, nlags=ACF_NLAGS,
        n_boot=ACF_N_BOOT, use_abs=True, seed=BOOTSTRAP_SEED, distance="weighted")
    best_L = bl["best_L"]
    print("Suggested avg_block_len:", best_L)
    display(bl["scores"])
else:
    best_L = AVG_BLOCK_LEN
    print("Using fixed avg_block_len:", best_L)

## Politis-Romano stationary bootstrap

Resamples the **common-sample** factor-ETF panel (and, when costs are on, the cost panel and the benchmark jointly, with shared indices). Inside every replica it rebuilds the signal, the weight schedule, the gross/net backtests and the metrics.

The headline robustness statistic is the **paired difference** (e.g. `MomTop1_80_20 - EW` and `MomTop1_80_20 - SPY`) computed within each replica, together with `P(strat > bench)`.

In [ ]:
PAIRED_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

if RUN_BOOTSTRAP:
    boot_primary = bootstrap_factor_mom_metrics(
        r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_daily_main,
        cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
        bench_returns=bench_ret, bench_name=BENCH_TICKER,
        n_boot=N_BOOT, avg_block_len=best_L, seed=BOOTSTRAP_SEED, rf_daily=rf_daily)

    print("Actual backtest metrics — gross")
    display(boot_primary["actual_metrics_gross"].round(4))
    print(f"Actual backtest metrics — net ({TC_ESTIMATOR.upper()})")
    display(boot_primary["actual_metrics_net"].round(4))

    for label, key in [("net CAGR", "CAGR"), ("net Sharpe", "ShR"),
                       ("net Martin", "Martin"), ("net Max drawdown", "MxDD")]:
        print(f"Bootstrap distribution — {label} ({TC_ESTIMATOR.upper()})")
        display(boot_primary["bootstrap_tables_net"][key].round(4))

    # ---- paired difference distributions: this is the real robustness test ----
    mats_net = boot_primary["simulated_metric_vectors_net"]
    print("Paired bootstrap difference — MomTop1_80_20 minus EW (net)")
    display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop1_80_20", "EW").round(4))
    print("Paired bootstrap difference — MomTop2_35_35 minus EW (net)")
    display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop2_35_35", "EW").round(4))
    if bench_ret is not None:
        print(f"Paired bootstrap difference — MomTop1_80_20 minus {BENCH_TICKER} (net)")
        display(paired_difference_summary(mats_net, PAIRED_METRICS, "MomTop1_80_20", BENCH_TICKER).round(4))
    print("Note: P(strat>bench) is the share of replicas where the strategy beats the "
          "benchmark on each metric (higher-is-better convention; for MxDD 'higher' = shallower).")

    # ---- AR vs CS vs FIXED on net metrics ----
    if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
        boot_comp = {}
        for est in estimators_requested:
            boot_comp[est] = bootstrap_factor_mom_metrics(
                r_raw, variant=VARIANT, cost_oneway_daily=cost_oneway_panels[est],
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG,
                bench_returns=bench_ret, bench_name=BENCH_TICKER,
                n_boot=N_BOOT, avg_block_len=best_L, seed=BOOTSTRAP_SEED, rf_daily=rf_daily)
        for label, key in [("CAGR", "CAGR"), ("Sharpe", "ShR"), ("Martin", "Martin")]:
            print(f"Net bootstrap distribution — {label} (AR vs CS vs FIXED)")
            display(pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"][key] for est in boot_comp},
                names=["Estimator", "Strategy"]).round(4))

## Random-selection (signal-permutation) null

Equal-weighting the five factor ETFs is itself an active factor bet, so beating EW conflates the momentum *signal* with differences in static factor exposure. This section builds a benchmark of strategies that are **structurally identical** to the real one (same 80/20 or 35/35 concentration, same rebalance dates, same costs, same timing) but pick the held factor(s) **at random** each month. The one-sided p-value is the share of random strategies that match or beat the real strategy on each metric: a small p-value is evidence that the **signal**, not the structure, drives performance.

In [ ]:
if RUN_RANDOM_NULL:
    # Structurally-identical, signal-free benchmark: random factor selection with the
    # SAME concentration, rebalance dates, costs and timing as the real strategy.
    # The one-sided p-value is the share of random strategies that match or beat the
    # real one -> isolates the contribution of the momentum SIGNAL from static exposure.
    NULL_METRICS = ("CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD")

    null_top1 = random_selection_null(
        r_common, signal, w_random_top1_80_20,
        cost_oneway_daily=cost_oneway_daily_main, cost_lookback_days=TC_LOOKBACK_DAYS,
        cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        implementation_lag=IMPLEMENTATION_LAG, metrics=NULL_METRICS,
        use_net=RUN_TRANSACTION_COSTS, n_random=N_RANDOM, seed=RANDOM_NULL_SEED, rf_daily=rf_daily)

    null_top2 = random_selection_null(
        r_common, signal, w_random_top2_35_35,
        cost_oneway_daily=cost_oneway_daily_main, cost_lookback_days=TC_LOOKBACK_DAYS,
        cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        implementation_lag=IMPLEMENTATION_LAG, metrics=NULL_METRICS,
        use_net=RUN_TRANSACTION_COSTS, n_random=N_RANDOM, seed=RANDOM_NULL_SEED, rf_daily=rf_daily)

    actual_net = perf_table(rets_net, rf_daily=rf_daily)
    print("Random-selection null — MomTop1_80_20 vs random top-1 (same 80/20 structure)")
    display(null_pvalues(actual_net.loc["MomTop1_80_20"], null_top1, NULL_METRICS).round(4))
    print("Random-selection null — MomTop2_35_35 vs random top-2 (same 35/35 structure)")
    display(null_pvalues(actual_net.loc["MomTop2_35_35"], null_top2, NULL_METRICS).round(4))

## Alternative weighting: separating selection from concentration

The fixed 80/20 and 35/35 schemes bundle two decisions — *which* factors to hold and *how concentrated* to be. Re-running the same momentum **selection** with **inverse-volatility** sizing isolates the selection effect from the concentration effect; `InvVol_all` is also a more neutral, risk-parity-like blend than equal weight (which ignores the factors' very different volatilities).

In [ ]:
if RUN_ALT_WEIGHTING:
    # Inverse-volatility sizing separates SELECTION from CONCENTRATION: same momentum
    # picks, but risk-based weights instead of the fixed 80/20 or 35/35. w_inverse_vol_all
    # is also a more neutral (risk-parity-like) blend than equal weight.
    w_iv_top1 = w_topk_inverse_vol(signal, r_common, k=1, lookback=ALT_WEIGHT_LOOKBACK)
    w_iv_top2 = w_topk_inverse_vol(signal, r_common, k=2, lookback=ALT_WEIGHT_LOOKBACK)
    w_iv_all = w_inverse_vol_all(signal, r_common, lookback=ALT_WEIGHT_LOOKBACK)
    alt = {}
    for nm, wm in [("MomTop1_invvol", w_iv_top1), ("MomTop2_invvol", w_iv_top2),
                   ("InvVol_all", w_iv_all)]:
        _, n = run_named_strategy(
            r_common, wm, cost_oneway_daily=cost_oneway_daily_main,
            cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
            charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        alt[nm] = n
    compare = pd.concat([rets_net[["MomTop1_80_20", "MomTop2_35_35", "EW"]],
                         pd.concat(alt, axis=1)], axis=1)
    print("Net metrics — fixed concentration (80/20, 35/35, EW) vs inverse-volatility sizing:")
    display(perf_table(compare, rf_daily=rf_daily).round(4))

## Serial-dependence diagnostics (does the bootstrap engine matter?)

Ljung-Box on **levels** captures the weak autoregressive / short-term-reversal structure in the mean; Ljung-Box on **squares** and the **ARCH-LM** test capture volatility clustering. Strong clustering is the case where the stationary bootstrap (which shuffles short blocks) understates long, clustered drawdowns — and where the **FHS engine** below is the more honest choice.

In [ ]:
if RUN_DEPENDENCE_DIAGNOSTICS:
    # If squared returns are strongly autocorrelated (ARCH), the stationary bootstrap
    # with short blocks understates clustered crashes -> prefer the FHS engine below.
    # Levels probe the (weak) AR / short-term-reversal structure in the mean.
    diag_series = {
        "MomTop1_net": rets_net["MomTop1_80_20"],
        "EW_net": rets_net["EW"],
        "Mom_minus_EW": (rets_net["MomTop1_80_20"] - rets_net["EW"]).dropna(),
    }
    if bench_ret is not None:
        diag_series[BENCH_TICKER] = bench_ret.reindex(rets_net.index).dropna()
    diag = dependence_diagnostics(diag_series, lags=DIAG_LAGS)
    print(f"Serial-dependence diagnostics (lags={DIAG_LAGS}) — Ljung-Box (levels & squares) + ARCH-LM:")
    display(diag.round(4))
    print("Reading: small LB_levels_p -> autocorrelation in the mean (AR / reversal); "
          "small LB_squares_p / ARCH_LM_p -> volatility clustering, which motivates the FHS engine.")

## Spanning regression (is the strategy just static factor exposure?)

Regressing the strategy's excess return on the five factor ETFs' excess returns (plus `SPY`) with **Newey-West** standard errors gives the cleanest benchmark of all: a **zero alpha** means the strategy is *spanned* by a fixed combination of the factors, so the timing/selection adds nothing beyond static exposure. This also sidesteps the `SPY` vs `SPY − rf` question, since everything is in excess returns.

In [ ]:
if RUN_SPANNING:
    # Regress the strategy's EXCESS return on the five factor ETFs' excess returns (+ SPY),
    # Newey-West HAC errors. An alpha indistinguishable from zero means the strategy is
    # "spanned" by a static factor combination: the timing/selection adds nothing.
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        sp = spanning_regression(rets_net[strat], r_common, rf_daily=rf_daily,
                                 extra_returns=bench_ret)
        print(f"Spanning regression — {strat} (net): Newey-West lags={sp['nw_lags']}, n={sp['n']}")
        print(f"  alpha (annualised) = {sp['alpha_annual']:.4f}   t = {sp['alpha_t']:.2f}   "
              f"p = {sp['alpha_p']:.3f}   R^2 = {sp['r2']:.3f}")
        display(sp["coefficients"].round(4))

## Sharpe-ratio inference: Ledoit-Wolf delta-Sharpe and PSR

The **Ledoit-Wolf (2008)** test compares two Sharpe ratios under non-iid returns (HAC standard error plus a studentized circular-block bootstrap), which is the correct way to ask whether `MomTop1_80_20` beats `EW`/`SPY` on a *risk-adjusted* basis. The **Probabilistic Sharpe Ratio** reports the probability the true Sharpe exceeds zero, correcting for skewness and fat tails. The search-corrected **Deflated Sharpe** appears in the multiple-testing section.

In [ ]:
if RUN_SHARPE_INFERENCE:
    # Ledoit-Wolf (2008): HAC + studentized circular-block bootstrap test for the
    # DIFFERENCE of two Sharpe ratios under non-iid returns (both series in excess units).
    def _exc(s):
        s = pd.Series(s).dropna()
        return s - rf_daily.reindex(s.index)
    benches = {"EW": rets_net["EW"]}
    if bench_ret is not None:
        benches[BENCH_TICKER] = bench_ret
    for bname, bser in benches.items():
        lw = sharpe_diff_test(_exc(rets_net["MomTop1_80_20"]), _exc(bser),
                              n_boot=LW_N_BOOT, seed=BOOTSTRAP_SEED)
        print(f"Delta-Sharpe (Ledoit-Wolf) — MomTop1_80_20 minus {bname}: "
              f"dSR_ann = {lw['delta_ann']:.3f}   t = {lw['tstat']:.2f}   p = {lw['pval']:.3f}   "
              f"(HAC bw = {lw['hac_bw']}, block = {lw['block_size']}, boot = {lw['n_boot']})")
    psr = probabilistic_sharpe_ratio(rets_net["MomTop1_80_20"], sr_benchmark=0.0)
    print(f"Probabilistic Sharpe (MomTop1_80_20 vs SR=0): PSR = {psr['psr']:.4f}  "
          f"(skew = {psr['skew']:.2f}, kurtosis = {psr['kurtosis']:.2f}, SR_ann = {psr['sharpe_ann']:.3f})")
    print("Deflated Sharpe (corrected for the specification search) is reported in the multiple-testing cell.")

## Break-even transaction cost

A single, robust summary of cost sensitivity: the flat one-way cost (in bps) at which the edge over `EW`/`SPY` disappears. Because it is a level rather than a path, it sidesteps the noisy AR/CS spread estimates — compare it against the per-trade costs estimated earlier.

In [ ]:
if RUN_BREAKEVEN:
    # The flat one-way cost (bps) that drives the edge to zero -- a single, interpretable
    # number that sidesteps the noisy AR/CS spread estimates.
    be_rows = {}
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        for mode, label in [("cagr_vs_ew", "CAGR vs EW"), ("sharpe_vs_ew", "Sharpe vs EW")]:
            be = breakeven_cost_bps(
                r_raw, variant=VARIANT, strat=strat, mode=mode, rf_daily=rf_daily,
                c_max_bps=BREAKEVEN_C_MAX_BPS, implementation_lag=IMPLEMENTATION_LAG,
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
            be_rows[f"{strat} | {label}"] = be.get("breakeven_bps", float("nan"))
        if bench_ret is not None:
            be = breakeven_cost_bps(
                r_raw, variant=VARIANT, strat=strat, mode="cagr_vs_spy", bench_returns=bench_ret,
                rf_daily=rf_daily, c_max_bps=BREAKEVEN_C_MAX_BPS, implementation_lag=IMPLEMENTATION_LAG,
                cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
            be_rows[f"{strat} | CAGR vs {BENCH_TICKER}"] = be.get("breakeven_bps", float("nan"))
    print(f"Break-even one-way cost in bps (edge vanishes at/above this cost; cap = {BREAKEVEN_C_MAX_BPS}):")
    display(pd.Series(be_rows, name="breakeven_oneway_bps").to_frame().round(2))
    print("Compare against the estimated per-trade costs printed earlier (AR/CS/FIXED).")

## BCa intervals on the realized statistics

Bias-corrected and accelerated intervals (circular-block bootstrap + block jackknife) for the realized Sharpe and CAGR. For asymmetric statistics these are better calibrated than the raw percentile intervals reported in the stationary-bootstrap section.

In [ ]:
if RUN_BCA:
    # Bias-corrected & accelerated intervals on the realized P&L (circular-block bootstrap
    # + block jackknife). Prefer PPW/arch for the block length; fall back to ACF matching
    # so the notebook does not fail if `arch` is not available in a lightweight runtime.
    try:
        bca_block = max(2, int(round(optimal_block_length_ppw(rets_net["MomTop1_80_20"])["stationary"])))
        block_method = "PPW / arch"
    except Exception as e:
        print(f"PPW block-length selection unavailable ({type(e).__name__}); using ACF-matching fallback.")
        try:
            _bl = choose_block_length_by_acf_matching(
                rets_net["MomTop1_80_20"], candidates=ACF_CANDIDATES, nlags=ACF_NLAGS,
                n_boot=ACF_N_BOOT, use_abs=True, seed=BOOTSTRAP_SEED, distance="weighted")
            bca_block = max(2, int(_bl["best_L"]))
            block_method = "ACF matching"
        except Exception as e2:
            print(f"ACF fallback failed ({type(e2).__name__}); using block size 10.")
            bca_block = 10
            block_method = "fixed fallback"

    print(f"BCa 95% intervals for MomTop1_80_20 (net): block size = {bca_block}, method = {block_method}, n_boot = {BCA_N_BOOT}")
    bca_rows = {}
    for mname, mfn in [("Sharpe", _metric_sharpe), ("CAGR", _metric_cagr)]:
        ci = bca_interval(rets_net["MomTop1_80_20"], mfn, block_size=bca_block,
                          n_boot=BCA_N_BOOT, alpha=0.05, seed=BOOTSTRAP_SEED)
        bca_rows[mname] = {"estimate": ci["theta"], "lo95": ci["lo"], "hi95": ci["hi"]}
    display(pd.DataFrame(bca_rows).T.round(4))

## Regime / subsample stability

Momentum is regime-dependent, so a single full-sample number can hide where the edge actually lives. Performance is broken out by calendar year, by sample halves, and by the trailing **market-volatility regime** (using `SPY`'s realized vol as the regime variable; swap in VIX if available).

In [ ]:
if RUN_REGIME:
    target = "MomTop1_80_20"
    print(f"{target} (net) — performance by calendar year:")
    display(metrics_by_calendar_year(rets_net[target], rf_daily=rf_daily).round(4))
    print(f"{target} (net) — performance by sample halves:")
    display(metrics_two_halves(rets_net[target], rf_daily=rf_daily).round(4))
    if bench_ret is not None:
        print(f"{target} (net) — by market volatility regime (trailing {REGIME_VOL_LOOKBACK}d vol of {BENCH_TICKER}):")
        display(metrics_by_vol_regime(rets_net[target], bench_ret, lookback=REGIME_VOL_LOOKBACK,
                                      n_regimes=REGIME_N, rf_daily=rf_daily).round(4))

## Dependence-aware bootstrap engines (FHS and VAR-sieve)

These engines address the concern that the stationary bootstrap, by reshuffling short blocks, breaks the volatility clustering and clustered crashes that drive momentum's left tail. **FHS** filters each factor with an AR(p)-GJR-GARCH(1,1), resamples the standardized-residual **rows** (preserving contemporaneous cross-factor correlation) and re-inflates through each series' own volatility recursion. The **VAR-sieve wild bootstrap** preserves cross-factor lead-lag and is robust to heteroskedasticity. Output mirrors the stationary bootstrap, so the same paired-difference reading applies. *FHS requires the `arch` package.*

In [ ]:
if RUN_ENGINE_BOOTSTRAP:
    # Dependence-aware engines that GENERATE panels (rather than resampling blocks):
    #   * FHS = AR(p)-GJR-GARCH filtering + row-resampled standardized residuals,
    #           preserving volatility clustering and fat (asymmetric) left tails;
    #   * VAR-sieve = VAR(p) + recursive WILD bootstrap, preserving cross-factor lead-lag.
    # The whole strategy is rebuilt inside every replica (costs via a flat-bps panel,
    # since simulated panels carry no OHLC). Requires the `arch` package for FHS.
    engine_boot = {}
    for eng in ENGINES:
        engine_boot[eng] = bootstrap_metrics_via_engine(
            r_common, engine=eng, variant=VARIANT, bench_returns=bench_ret, bench_name=BENCH_TICKER,
            flat_cost_oneway_bps=(ENGINE_FLAT_COST_BPS if RUN_TRANSACTION_COSTS else None),
            ar_lags=ENGINE_AR_LAGS, var_lags=ENGINE_VAR_LAGS,
            n_sims=ENGINE_N_SIMS, seed=ENGINE_SEED, rf_daily=rf_daily,
            implementation_lag=IMPLEMENTATION_LAG)
    for eng in ENGINES:
        mats = engine_boot[eng]["simulated_metric_vectors_net"]
        print(f"[{eng.upper()}] net Sharpe distribution")
        display(engine_boot[eng]["bootstrap_tables_net"]["ShR"].round(4))
        print(f"[{eng.upper()}] net Martin distribution")
        display(engine_boot[eng]["bootstrap_tables_net"]["Martin"].round(4))
        print(f"[{eng.upper()}] paired difference MomTop1_80_20 - EW (net)")
        display(paired_difference_summary(mats, PAIRED_METRICS, "MomTop1_80_20", "EW").round(4))
        if bench_ret is not None:
            print(f"[{eng.upper()}] paired difference MomTop1_80_20 - {BENCH_TICKER} (net)")
            display(paired_difference_summary(mats, PAIRED_METRICS, "MomTop1_80_20", BENCH_TICKER).round(4))
    print("Compare these intervals with the stationary-bootstrap ones: if the edge shrinks "
          "under FHS, it depended on the volatility clustering the stationary bootstrap broke up.")

## Multiple testing under specification search

Reporting the best of several specifications inflates significance. **White's (2000) Reality Check** and **Hansen's (2005) SPA** test the null that the best specification does **not** beat the benchmark once the whole search (variant × concentration × cost) is accounted for; the **Deflated Sharpe Ratio (Bailey & Lopez de Prado 2014)** corrects the best spec's Sharpe for the number of trials and for non-normality. These directly answer the data-snooping objection.

In [ ]:
if RUN_MULTIPLE_TESTING:
    # Across the grid (variant x concentration x cost), is the BEST specification really
    # better than the benchmark once the search is accounted for?  White (2000) Reality
    # Check + Hansen (2005) SPA, plus the Deflated Sharpe Ratio on the best spec.
    mt_cost_panels = {"GROSS": None}
    if RUN_TRANSACTION_COSTS:
        for est in estimators_requested:
            mt_cost_panels[est] = cost_oneway_panels[est]
    spec_mat, sr_trials = build_spec_return_matrix(
        r_raw, variants=("6-1", "12-1"), concentrations=("top1", "top2"),
        cost_panels=mt_cost_panels, rf_daily=rf_daily, implementation_lag=IMPLEMENTATION_LAG,
        cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
    bench_for_mt = rets_net["EW"]
    rc = reality_check_white(spec_mat, bench_for_mt, block_size=MT_BLOCK_SIZE, n_boot=MT_N_BOOT, seed=MT_SEED)
    spa = spa_test_hansen(spec_mat, bench_for_mt, block_size=MT_BLOCK_SIZE, reps=MT_N_BOOT, seed=MT_SEED)
    print(f"Specification grid: K = {spec_mat.shape[1]} strategies vs EW benchmark, n = {rc['n']}")
    print(f"White Reality Check   p = {rc['pvalue']:.4f}   (best spec: {rc['best_spec']})")
    print(f"Hansen SPA p-values:  " + ", ".join(f"{k}={v:.4f}" for k, v in spa['pvalues'].items())
          + "   (quote the 'consistent' one)")
    print("Mean annualised excess-over-EW by specification:")
    display(rc["mean_excess_ann"].round(4).to_frame("mean_excess_ann"))
    dsr = deflated_sharpe_ratio(spec_mat[rc["best_spec"]], list(sr_trials.values()))
    print(f"Deflated Sharpe Ratio (best spec '{rc['best_spec']}'): DSR = {dsr['dsr']:.4f}   "
          f"SR_ann = {dsr['sharpe_ann']:.3f} vs threshold SR0_ann = {dsr['sr0_threshold_ann']:.3f}   "
          f"(N_trials = {dsr['n_trials']})")
    _sr_std = float(np.std([s for s in sr_trials.values() if np.isfinite(s)], ddof=1))
    print(f"  CAVEAT: the {spec_mat.shape[1]} specs are highly correlated (per-obs trial-Sharpe "
          f"std = {_sr_std:.4f}); the DSR's independent-trials assumption is then violated and it "
          f"UNDER-deflates (SR0 collapses toward 0). Treat the SPA 'consistent' p-value above as "
          f"the multiplicity-robust verdict, not the DSR.")

## Long-short academic factors: localising the gap

The decisive economic test. Factor momentum is documented on **long-short** factor returns (Gupta-Kelly; Ehsani-Linnainmaa), not on long-only ETF wrappers. This section runs the *same* signal on the Fama-French long-short factors (SMB, HML, RMW, CMA, Mom). If momentum beats the equal-weight factor blend on the **long-short** factors while it is spanned in the **ETF** version, the implementation gap is the *vehicle* (market beta plus a thin long-only tilt), not the signal. Requires network access to Kenneth French's data library (or a fallback CSV via `LONGSHORT_CSV_PATH`).

In [ ]:
if RUN_LONGSHORT:
    # Localise the gap: run the SAME cross-sectional momentum signal on the LONG-SHORT
    # academic factors (Fama-French SMB/HML/RMW/CMA/Mom). If momentum works there but is
    # spanned in the ETF wrappers, the gap is the VEHICLE (market beta + thin long-only tilt),
    # not the signal. Requires network access to French's library (or a fallback CSV).
    ls_rets = ls_rf = ls_mkt = None
    try:
        ls_rets, ls_rf, ls_mkt = download_famafrench_factors(start=START_DATE)
        print(f"Fama-French long-short factors: {ls_rets.shape}, "
              f"{ls_rets.index.min().date()} to {ls_rets.index.max().date()} "
              f"({', '.join(ls_rets.columns)})")
    except Exception as e:
        if LONGSHORT_CSV_PATH:
            ls_rets, ls_rf, ls_mkt = load_factor_returns_csv(LONGSHORT_CSV_PATH, rf_col="RF", mkt_col="Mkt-RF")
            print(f"FF download failed ({type(e).__name__}); loaded factors from {LONGSHORT_CSV_PATH}.")
        else:
            print(f"Fama-French factors unavailable ({type(e).__name__}). Set LONGSHORT_CSV_PATH "
                  f"to a CSV of daily factor returns to run this section.")

    if ls_rets is not None:
        ls_rf_use = ls_rf if ls_rf is not None else rf_daily
        ls_common = ls_rets.dropna(how="any")
        ls_out = run_factor_mom_on_panel(ls_rets, variant=VARIANT,
                                         implementation_lag=IMPLEMENTATION_LAG, cost_oneway_bps=LONGSHORT_COST_BPS)
        ls_tab = perf_table(ls_out["rets"], rf_daily=ls_rf_use)
        print("Long-short factor-momentum metrics:")
        display(ls_tab.round(4))

        # paired bootstrap MomTop1 - EW on the long-short panel (rebuilt per replica)
        LSM = ("CAGR", "ShR", "Martin", "MxDD")
        mats = {(m, s): [] for m in LSM for s in ("MomTop1", "EW")}
        T = len(ls_common)
        for ii in stationary_bootstrap_index_stream(T, best_L, N_BOOT, BOOTSTRAP_SEED):
            sim = ls_common.iloc[ii].copy(); sim.index = ls_common.index
            tb = perf_table(run_factor_mom_on_panel(sim, variant=VARIANT,
                            implementation_lag=IMPLEMENTATION_LAG)["rets"], rf_daily=ls_rf_use)
            for m in LSM:
                for s in ("MomTop1", "EW"):
                    mats[(m, s)].append(tb.loc[s, m] if (s in tb.index and m in tb.columns) else np.nan)
        mats = {k: np.asarray(v, float) for k, v in mats.items()}
        print("Long-short paired bootstrap — MomTop1 minus EW:")
        display(paired_difference_summary(mats, LSM, "MomTop1", "EW").round(4))

        # signal-permutation null on the long-short panel (same 80/20 structure, random pick)
        rngp = np.random.default_rng(RANDOM_NULL_SEED)
        null = {m: [] for m in LSM}
        for _ in range(N_RANDOM):
            wr = w_random_top1_80_20(ls_out["signal"].index, list(ls_out["signal"].columns), rngp)
            mm = perf_metrics(backtest_returns_panel_monthly(ls_common, wr,
                              implementation_lag=IMPLEMENTATION_LAG), rf_daily=ls_rf_use)
            for m in LSM:
                null[m].append(mm.get(m, np.nan))
        null = {m: np.asarray(v, float) for m, v in null.items()}
        print("Long-short signal-permutation null — MomTop1 vs random top-1:")
        display(null_pvalues(ls_tab.loc["MomTop1"], null, LSM).round(4))

        # side-by-side: ETF wrappers vs long-short factors (the localisation of the gap)
        etf_tab = perf_table(rets_net, rf_daily=rf_daily)
        cmp = pd.DataFrame({
            "ETF_MomTop1": etf_tab.loc["MomTop1_80_20", ["CAGR", "ShR", "Martin"]],
            "ETF_EW": etf_tab.loc["EW", ["CAGR", "ShR", "Martin"]],
            "LS_MomTop1": ls_tab.loc["MomTop1", ["CAGR", "ShR", "Martin"]],
            "LS_EW": ls_tab.loc["EW", ["CAGR", "ShR", "Martin"]],
        })
        print("ETF wrappers vs long-short factors:")
        display(cmp.round(4))
        print("If LS_MomTop1 beats LS_EW while ETF_MomTop1 does not, the edge lives in the "
              "long-short factor construction, not in the long-only ETF implementation.")

## True out-of-sample walk-forward

A different *kind* of evidence than the in-sample SPA test: an expanding-window protocol that re-picks the best specification on past data only and banks its future returns. If the OOS-selected strategy does not beat equal-weight on the same window, the specification search does not survive out of sample.

In [ ]:
if RUN_WALK_FORWARD:
    # True OOS: expanding window, re-pick the best spec by IN-SAMPLE Sharpe ~quarterly, then
    # bank that spec's next-quarter OOS return. Answers the spec-search-overfit worry directly.
    if "spec_mat" not in globals():
        _mt_panels = {"GROSS": None}
        if RUN_TRANSACTION_COSTS:
            for est in estimators_requested:
                _mt_panels[est] = cost_oneway_panels[est]
        spec_mat, _ = build_spec_return_matrix(
            r_raw, variants=("6-1", "12-1"), concentrations=("top1", "top2"),
            cost_panels=_mt_panels, rf_daily=rf_daily, implementation_lag=IMPLEMENTATION_LAG,
            cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG, charge_initial_trade=TC_CHARGE_INITIAL_TRADE)
    wf = walk_forward_select(spec_mat, min_train_days=WF_MIN_TRAIN_DAYS,
                             reselect_every_days=WF_RESELECT_EVERY_DAYS, rf_daily=rf_daily)
    print(f"Walk-forward OOS: {wf['n_reselections']} reselections, n = {wf['oos_metrics'].get('n')}")
    ew_oos = perf_metrics(rets_net["EW"].reindex(wf["oos_returns"].index), rf_daily=rf_daily)
    comp = pd.DataFrame({"OOS_selected": pd.Series(wf["oos_metrics"]),
                         "EW_same_window": pd.Series(ew_oos)}).T[
        ["CAGR", "ShR", "SoR", "Calmar", "Martin", "MxDD"]]
    display(comp.round(4))
    print("Specifications chosen over time:")
    display(wf["chosen"]["spec"].value_counts().to_frame("times_selected"))
    print("If the OOS-selected line does not beat EW on the same window, the spec search does "
          "not add value out-of-sample (consistent with the SPA result).")

## Rank / quantile weighting (and extended-universe hook)

Inverse-volatility sizing (earlier) and **rank/quantile** weighting here separate *selection* from the arbitrary 80/20 concentration. The builders scale to any universe size; set `EXTENDED_TICKERS` (8-12 distinct single-factor ETFs chosen by a pre-committed rule) and re-run from the data cell to test whether the negative result is an artefact of `N = 5`.

In [ ]:
if RUN_QUANTILE_WEIGHTING:
    # Rank/quantile sizing separates SELECTION from the fixed 80/20 concentration and is the
    # natural construction for a larger universe.
    qw = {"MomQ_top": w_quantile_long(signal, top_frac=QUANTILE_TOP_FRAC),
          "MomRank": w_rank_proportional(signal),
          "EW": w_equal(signal)}
    qres = {}
    for nm, wm in qw.items():
        _, nnet = run_named_strategy(r_common, wm, cost_oneway_daily=cost_oneway_daily_main,
                                     cost_lookback_days=TC_LOOKBACK_DAYS, cost_agg=TC_AGG,
                                     charge_initial_trade=TC_CHARGE_INITIAL_TRADE, implementation_lag=IMPLEMENTATION_LAG)
        qres[nm] = nnet
    print(f"Rank/quantile weighting on the 5-ETF universe (top_frac={QUANTILE_TOP_FRAC}, net):")
    display(perf_table(pd.concat(qres, axis=1), rf_daily=rf_daily).round(4))
    if EXTENDED_TICKERS:
        print(f"EXTENDED_TICKERS set ({len(EXTENDED_TICKERS)} ETFs). For a full extended-universe run, "
              f"set TICKERS = EXTENDED_TICKERS at the top and re-run from the data cell; the rank/"
              f"quantile builders above scale to any N.")
    else:
        print("To test breadth, set EXTENDED_TICKERS to 8-12 distinct single-factor ETFs (by a "
              "pre-committed rule), set TICKERS = EXTENDED_TICKERS, and re-run. Builders scale to any N.")

## Conditional (regime) spanning and rolling alpha

The full-sample spanning alpha is zero, but momentum is regime-dependent. A two-alpha spanning regression (interacting the constant and factor loadings with a high/low market-volatility dummy, Newey-West) tests whether a regime-conditional alpha is hidden inside the unconditional zero; the rolling 2-year alpha shows whether any alpha is episodic.

In [ ]:
if RUN_CONDITIONAL_SPANNING:
    # Does the zero full-sample alpha hide a regime-conditional alpha? Two-alpha spanning with
    # a high/low market-volatility dummy (Newey-West), plus the rolling alpha for inspection.
    reg = vol_regime_dummy(bench_ret if bench_ret is not None else rets_net["EW"],
                           rets_net.index, lookback=COND_VOL_LOOKBACK, q=0.5)
    for strat in ["MomTop1_80_20", "MomTop2_35_35"]:
        csp = spanning_regression_conditional(rets_net[strat], r_common, reg,
                                              rf_daily=rf_daily, extra=bench_ret)
        print(f"Conditional spanning — {strat} (net): NW lags={csp['nw_lags']}, "
              f"n_low={csp['n_low']}, n_high={csp['n_high']}")
        display(pd.DataFrame({"low_vol": csp["low_regime"], "high_vol": csp["high_regime"],
                              "high_minus_low": csp["delta_high_minus_low"]}).T.round(4))
    ra = rolling_spanning_alpha(rets_net["MomTop1_80_20"], r_common, rf_daily=rf_daily,
                                window=504, extra=bench_ret)
    ax = ra.plot(figsize=(11, 4), title="MomTop1_80_20 — rolling 2y annualised spanning alpha (net)")
    ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("alpha (annualised)"); plt.show()

## Spanning decomposition and the mechanism behind the positive finding

Two complementary views. The **static replication** rebuilds MomTop1 from its fixed spanning betas (alpha excluded) and reports how tightly that constant factor blend tracks the strategy — making "spanned" concrete. The **selection-quality** diagnostic asks *why* momentum beats the random-selection null on drawdowns but not on the mean: it ranks the forward return of the picked factor among all factors and splits by regime. A pick that avoids the worst factor (especially in high-volatility months) is a *defensive* edge — it improves the tail, not the average.

In [ ]:
if RUN_SELECTION_MECHANISM:
    # (1) Static replication: MomTop1 is (approximately) a FIXED factor blend.
    rep = static_replication(rets_net["MomTop1_80_20"], r_common, rf_daily=rf_daily, extra=bench_ret)
    print(f"Static replication of MomTop1_80_20: corr(static, actual) = {rep['corr_static_vs_actual']:.3f}, "
          f"R^2 = {rep['r2']:.3f}, alpha_ann = {rep['alpha_ann']:.4f}; "
          f"static Sharpe {rep['static_sharpe_ann']:.3f} vs actual {rep['actual_sharpe_ann']:.3f}")
    print("Implied long-only factor weights of the static replica:")
    display(rep["implied_long_weights"].round(3).to_frame("weight"))

    # (2) Mechanism for the positive finding: does momentum AVOID the worst factor, more so
    #     in high-volatility months?
    reg = vol_regime_dummy(bench_ret if bench_ret is not None else rets_net["EW"],
                           r_common.index, lookback=COND_VOL_LOOKBACK, q=0.5)
    sq = selection_quality_by_regime(signal, r_common, regime=reg, k=1)
    print("Selection quality of the top-1 momentum pick (rank 1 = worst .. 5 = best):")
    display(sq["summary"].round(4))
    print("If P_worst sits below the random 1/N and the mean rank above (N+1)/2 — more so in "
          "high-vol — the edge is DEFENSIVE: momentum avoids the worst factor, improving drawdowns, "
          "which matches the permutation null winning on Martin/Calmar but not on the mean.")